In [18]:
import os

from langchain.chat_models import init_chat_model
from dotenv import load_dotenv

# 1. Clear any existing tracing variables from active memory completely
if "LANGCHAIN_TRACING" in os.environ:
    del os.environ["LANGCHAIN_TRACING"]
if "LANGCHAIN_TRACING_V2" in os.environ:
    del os.environ["LANGCHAIN_TRACING_V2"]

# 2. Now load your clean API keys safely from your .env file
load_dotenv()
os.environ["LMSTUDIO_API_KEY"] =""
model = init_chat_model(
    model="qwen/qwen3-1.7b",
    model_provider="openai",                    # Use "openai" as the engine
    base_url="http://192.168.10.103:1234/v1",  # Keeps pointing to your local machine
    temperature=0.0
    )

message =model.invoke("WHATS THE WEATHER IN BIRATNAAGT")
print(message.content)




The weather in Biratnagar, Nepal, can vary significantly depending on the season and time of year. Here’s a general overview:

- **Seasonal Patterns**:  
  - **Monsoon Season (June–September)**: Heavy rainfall is common, with thunderstorms possible. The region experiences frequent rain showers, especially in June and July.  
  - **Dry Season (October–May)**: Cooler temperatures prevail, with occasional snowfall in higher elevations (e.g., the Himalayas). Rain is less frequent but can still occur.  

- **Current Conditions**:  
  - As of the latest available data (before this response), Biratnagar experienced **moderate rainfall** and **cool temperatures**, with some areas seeing light to moderate winds.  

For real-time updates, check a reliable weather service like the **Nepal Meteorological Department** or apps like **Weather.com**, **AccuWeather**, or **Meteo Italia**. Weather can change rapidly, especially during monsoons. Let me know if you'd like help finding specific forecasts

### SUMMARIZATION MIDDLE WARE

compresses the long history inorder to decrese the consumption of token and halucinations


In [121]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage

# 1. Connect to your local Qwen model
local_llm = ChatOpenAI(
model="qwen/qwen3-1.7b",
api_key="lm-studio",  
base_url="http://192.168.10.103:1234/v1",
temperature=0.0
)

# 2. The exact code from the tutorial, using your local model!
agent = create_agent(
model=local_llm,
checkpointer=InMemorySaver(),
middleware=[
    SummarizationMiddleware(
        model=local_llm,           # Uses Qwen to write the summaries
        trigger=("messages", 10),  # Triggers when 10 messages pile up
        keep=("messages", 4)       # Keeps the 4 most recent messages
    )
]
)

# 3. How to test it
# Because you are using InMemorySaver(), LangChain requires a "thread_id" 
# so it knows which conversation memory file to save this to.
config = {"configurable": {"thread_id": "session-1"}}
config2 = {"configurable": {"thread_id": "session-1"}}


print("--- Sending Message 1 ---")
response1 = agent.invoke(
{"messages": [HumanMessage(content="Hi, my name is Aaditya and I am learning ethical hacking.")]}, 
config=config
)
# The new agent returns a full list of messages; we print the content of the very last one
print("\nAgent Reply:", response1["messages"][-1].content)

print("\n--- Sending Message 2 ---")
response2 = agent.invoke(
{"messages": [HumanMessage(content="What is my name, and what am I trying to learn just write 1000 keywords its enough")]}, 
config=config2
)
print("\nAgent Reply:", response2["messages"][-1].content)

--- Sending Message 1 ---

Agent Reply: 

Hi Aaditya! Welcome to the exciting world of ethical hacking! Here's a structured guide to help you start your journey:

---

### **1. Understanding Ethical Hacking**
- **What is Ethical Hacking?**  
  It’s the practice of identifying and fixing security vulnerabilities in systems, networks, or applications to prevent malicious attacks. Ethical hackers (white hat) work with authorization to test defenses.

- **Why Learn It?**  
  To protect systems from real-world threats, understand attack vectors, and develop skills for penetration testing, network security, and incident response.

---

### **2. Key Concepts**
- **Types of Attacks**:  
  - **Phishing**, **SQL Injection**, **DDoS**, **Malware**, **Social Engineering**.
- **Penetration Testing Phases**:  
  1. Reconnaissance (scanning networks/hosts).  
  2. Scanning & Enumeration (identifying vulnerabilities).  
  3. Exploitation (taking advantage of flaws).  
  4. Post-Exploitation (gaining a